# Augmented Lagrangian Predictive Coding (PC-ALM) — MNIST Demo

This notebook implements the PC-ALM algorithm from:

> Seely & Gould, *"Augmented Lagrangian Predictive Coding"*, [arXiv:2605.31022](https://arxiv.org/abs/2605.31022)

PC-ALM augments standard PC inference with per-layer Lagrange multipliers (dual variables) that accumulate prediction errors across inference steps. At convergence, the duals recover exact backpropagation adjoints, closing the PC–BP gap even in deep narrow networks.

**What this notebook does:**
1. Trains the same architecture with **standard PC** (`InferenceSGD`) and **PC-ALM** (`InferenceALM`)
2. Compares accuracy and training speed
3. Demonstrates BP alignment in a small linear network

**Architecture:**
```
pixels(784) → hidden1(128) → hidden2(64) → class(10)
 Identity      ReLU           ReLU         Softmax+CE
```

## 1. Imports & Setup

In [ ]:
import jax
import jax.numpy as jnp
import optax
import time

from fabricpc.nodes import Linear, IdentityNode
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params
from fabricpc.graph_initialization.state_initializer import initialize_graph_state
from fabricpc.core.activations import IdentityActivation, ReLUActivation, SoftmaxActivation
from fabricpc.core.energy import CrossEntropyEnergy
from fabricpc.core.inference import InferenceSGD, InferenceALM, run_inference
from fabricpc.core.initializers import XavierInitializer
from fabricpc.core.learning import compute_local_weight_gradients_alm
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.utils.data.dataloader import MnistLoader
from fabricpc import setup_jax

setup_jax()
jax.config.update("jax_default_prng_impl", "threefry2x32")

## 2. Shared Hyperparameters

In [ ]:
NUM_EPOCHS = 10
BATCH_SIZE = 200
LR = 0.001
INFER_STEPS = 20
ETA_INFER = 0.05

## 3. Build the Network

We define a helper that builds the same MLP architecture with any inference algorithm. This lets us compare standard PC and PC-ALM on identical networks.

In [3]:
def build_structure(inference):
    """Build a 4-layer MLP with the given inference algorithm."""
    pixels = IdentityNode(shape=(784,), name="pixels")
    hidden1 = Linear(
        shape=(128,),
        activation=ReLUActivation(),
        name="hidden1",
        weight_init=XavierInitializer(),
    )
    hidden2 = Linear(
        shape=(64,),
        activation=ReLUActivation(),
        name="hidden2",
        weight_init=XavierInitializer(),
    )
    output = Linear(
        shape=(10,),
        activation=SoftmaxActivation(),
        energy=CrossEntropyEnergy(),
        name="class",
        weight_init=XavierInitializer(),
    )
    return graph(
        nodes=[pixels, hidden1, hidden2, output],
        edges=[
            Edge(source=pixels, target=hidden1.slot("in")),
            Edge(source=hidden1, target=hidden2.slot("in")),
            Edge(source=hidden2, target=output.slot("in")),
        ],
        task_map=TaskMap(x=pixels, y=output),
        inference=inference,
    )

## 4. Training Helper

In [ ]:
def run_experiment(name, inference, rng_key):
    """Train and evaluate a model, returning (accuracy, time_per_epoch)."""
    structure = build_structure(inference)
    graph_key, train_key, eval_key = jax.random.split(rng_key, 3)
    params = initialize_params(structure, graph_key)

    train_loader = MnistLoader(
        "train", batch_size=BATCH_SIZE, tensor_format="flat", shuffle=True, seed=42
    )
    test_loader = MnistLoader(
        "test", batch_size=BATCH_SIZE, tensor_format="flat", shuffle=False
    )
    optimizer = optax.adamw(LR, weight_decay=0.1)

    n_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"  {len(structure.nodes)} nodes, {len(structure.edges)} edges, {n_params:,} params")
    print(f"{'='*60}")

    start = time.time()
    trained_params, energy_history, _ = train_pcn(
        params=params,
        structure=structure,
        train_loader=train_loader,
        optimizer=optimizer,
        config={"num_epochs": NUM_EPOCHS},
        rng_key=train_key,
        verbose=True,
    )
    elapsed = time.time() - start

    metrics = evaluate_pcn(trained_params, structure, test_loader, {}, eval_key)
    acc = metrics["accuracy"] * 100
    avg_time = elapsed / NUM_EPOCHS

    print(f"\n  Test accuracy: {acc:.2f}%")
    print(f"  Avg time/epoch: {avg_time:.2f}s")
    return acc, avg_time, energy_history

## 5. Train with Standard PC

In [ ]:
master_key = jax.random.PRNGKey(0)
key_pc, key_alm = jax.random.split(master_key)

acc_pc, time_pc, energies_pc = run_experiment(
    "Standard PC (InferenceSGD)",
    InferenceSGD(eta_infer=ETA_INFER, infer_steps=INFER_STEPS),
    key_pc,
)

## 6. Train with PC-ALM

Same architecture and hyperparameters, but using `InferenceALM` with dual variables.

Key parameters:
- `alpha=1.0`: dual step size (how fast multipliers accumulate errors)
- `rho=1.0`: penalty strength (should match the energy precision, default 1.0)

In [ ]:
acc_alm, time_alm, energies_alm = run_experiment(
    "PC-ALM (InferenceALM, alpha=1.0, rho=1.0)",
    InferenceALM(
        eta_infer=ETA_INFER,
        infer_steps=INFER_STEPS,
        alpha=1.0,
        rho=1.0,
    ),
    key_alm,
)

## 7. Comparison Summary

In [ ]:
print(f"{'='*60}")
print(f"  {'Method':<30} {'Accuracy':>10} {'Time/epoch':>12}")
print(f"  {'-'*54}")
print(f"  {'Standard PC':<30} {acc_pc:>9.2f}% {time_pc:>10.2f}s")
print(f"  {'PC-ALM (alpha=1, rho=1)':<30} {acc_alm:>9.2f}% {time_alm:>10.2f}s")
print(f"{'='*60}")

## 8. BP Alignment Test

The paper's key theoretical result: in a linear network, PC-ALM weight gradients converge to exact backpropagation gradients.

We verify this by building a small linear chain, running PC-ALM inference, and comparing the resulting weight gradients against `jax.grad` of the forward loss.

In [8]:
# Build a small linear network: input(8) -> h0(8) -> output(8)
width = 8
batch_size = 4

alm_test = InferenceALM(eta_infer=0.05, infer_steps=200, alpha=1.0, rho=1.0)

inp = Linear(shape=(width,), name="input")
h0 = Linear(shape=(width,), activation=IdentityActivation(), name="h0")
out = Linear(shape=(width,), name="output")

structure_bp = graph(
    nodes=[inp, h0, out],
    edges=[
        Edge(source=inp, target=h0.slot("in")),
        Edge(source=h0, target=out.slot("in")),
    ],
    task_map=TaskMap(x=inp, y=out),
    inference=alm_test,
)

rng = jax.random.PRNGKey(42)
params_bp = initialize_params(structure_bp, rng)
x = jax.random.normal(rng, (batch_size, width))
y = jax.random.normal(jax.random.PRNGKey(99), (batch_size, width))

# Run PC-ALM inference
clamps = {structure_bp.task_map["x"]: x, structure_bp.task_map["y"]: y}
init_state = initialize_graph_state(
    structure_bp, batch_size, rng, clamps=clamps, params=params_bp
)
final_state = run_inference(params_bp, init_state, clamps, structure_bp)
grads_alm_bp = compute_local_weight_gradients_alm(
    params_bp, final_state, structure_bp, rho=1.0
)

# Compute true BP gradients
node_order = list(structure_bp.node_order)

def bp_loss(all_params):
    h = x
    for nn in node_order:
        ni = structure_bp.nodes[nn].node_info
        if ni.in_degree == 0:
            continue
        np_ = all_params.nodes[nn]
        wk = list(np_.weights.keys())[0]
        h = h @ np_.weights[wk]
        if np_.biases:
            bk = list(np_.biases.keys())[0]
            h = h + np_.biases[bk]
    return 0.5 * jnp.mean(jnp.sum((y - h) ** 2, axis=-1))

grads_true_bp = jax.grad(bp_loss)(params_bp)

# Compare
print(f"{'Node':<12} {'Weight key':<20} {'Cosine similarity':>18}")
print("-" * 52)
for nn in node_order:
    ni = structure_bp.nodes[nn].node_info
    if ni.in_degree == 0:
        continue
    for wk in grads_alm_bp.nodes[nn].weights:
        ga = grads_alm_bp.nodes[nn].weights[wk].flatten()
        gb = grads_true_bp.nodes[nn].weights[wk].flatten()
        cos = float(jnp.dot(ga, gb) / (jnp.linalg.norm(ga) * jnp.linalg.norm(gb) + 1e-10))
        print(f"{nn:<12} {wk:<20} {cos:>18.6f}")

Node         Weight key            Cosine similarity
----------------------------------------------------
h0           input->h0:in                   1.000000
output       h0->output:in                  1.000000


Cosine similarity of **1.0** confirms that PC-ALM weight gradients exactly match backpropagation in linear networks, as predicted by the theory (Theorem 1 of the paper).